# CrewAI Basics with Agentic Assistants

This notebook explores CrewAI integration with the Agentic Assistants framework.

## What is CrewAI?

CrewAI is a framework for orchestrating role-playing AI agents. It allows you to:
- Define agents with specific roles, goals, and backstories
- Create tasks for agents to complete
- Form crews that collaborate to achieve complex objectives

## Topics Covered

1. Creating agents with Ollama
2. Defining tasks
3. Building a crew
4. Running with observability
5. Analyzing results


In [ ]:
# Setup
from agentic_assistants import AgenticConfig, OllamaManager
from agentic_assistants.adapters import CrewAIAdapter
from agentic_assistants.utils.logging import setup_logging

setup_logging(level="INFO")

# Initialize
config = AgenticConfig()
ollama = OllamaManager(config)
adapter = CrewAIAdapter(config)

# Ensure Ollama is ready
ollama.ensure_running()
model = ollama.ensure_model()
print(f"Using model: {model}")


## Creating Agents

The adapter provides a convenient method for creating Ollama-powered agents:


In [ ]:
# Create a research agent
researcher = adapter.create_ollama_agent(
    role="Research Analyst",
    goal="Find and analyze information on given topics",
    backstory="""You are an expert research analyst with a keen eye for detail.
    You excel at finding relevant information and presenting it clearly.
    You always structure your findings logically.""",
    allow_delegation=False,
)

# Create a writer agent
writer = adapter.create_ollama_agent(
    role="Content Writer", 
    goal="Create clear and engaging content from research",
    backstory="""You are a skilled writer who transforms complex information
    into accessible, engaging content. You focus on clarity and impact.""",
    allow_delegation=False,
)

print(f"Created agents: {researcher.role}, {writer.role}")


## Defining Tasks

Tasks specify what work agents should complete:


In [ ]:
topic = "Benefits of local LLM deployment"

# Research task
research_task = adapter.create_task(
    description=f"""Research the topic: {topic}
    
    Provide:
    1. Overview of the concept
    2. Key benefits (at least 3)
    3. Potential challenges
    4. Best practices""",
    agent=researcher,
    expected_output="A structured research summary with clear sections",
)

# Writing task
writing_task = adapter.create_task(
    description=f"""Based on the research, write a brief article about {topic}.
    
    Requirements:
    - Engaging introduction
    - Clear explanation of benefits
    - Practical recommendations
    - Keep it under 300 words""",
    agent=writer,
    expected_output="A concise, well-written article",
)

print("Tasks created!")


## Building and Running the Crew

Now we assemble the crew and run it with full observability:


In [ ]:
# Create the crew
crew = adapter.create_crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=True,
)

print(f"Crew assembled with {len(crew.agents)} agents and {len(crew.tasks)} tasks")


In [ ]:
# Run with tracking
# Note: This may take a few minutes depending on your model and hardware

result = adapter.run_crew(
    crew,
    inputs={"topic": topic},
    experiment_name="crewai-notebook",
    run_name="research-crew-demo",
)

print("\n" + "="*60)
print("CREW OUTPUT:")
print("="*60)
print(result)


## What's Being Tracked?

When you run a crew with the adapter, the following is automatically tracked:

**MLFlow:**
- Run parameters (model, agent count, task count)
- Duration metrics
- Success/failure status
- Output artifacts

**OpenTelemetry:**
- Distributed traces for the crew run
- Spans for each agent interaction
- Latency metrics

View your experiments in MLFlow UI: http://localhost:5000
